<a href="https://colab.research.google.com/github/sultanjacob/Applied-Machine-Learning/blob/main/09_Ensemble_Margin_Classification/01_Cherry_Picker_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 9: Margin-Bleed Classification & Ensemble Arbitration

## The Business Problem: The "Cherry-Picker" Parasite Basket
Retailers use deep discounts (loss leaders) to drive foot traffic, assuming customers will fill the rest of their baskets with high-margin items. However, "cherry-pickers" exploit this by purchasing *only* the deeply discounted items. These transactions are parasitic, they generate negative margins and actively bleed profitability.

Our goal is to build a classification system to identify these transactions. Because penalizing a legitimate shopper is dangerous, we will not rely on a single model. We will build a **Hard Voting Ensemble** (Support Vector Machines, K-Nearest Neighbors, and Decision Trees) to act as a strict executive arbiter.

## Step 1: Target Definition
Before classification, we must engineer our target variable (`Y`). We will analyze the historical transactions to calculate the `Discount_Ratio` of every basket. By observing the distribution of these discounts, we will draw a strict mathematical boundary to label baskets as `Parasitic (1)` or `Profitable (0)`.

In [1]:
!pip install completejourney_py
import pandas as pd
import numpy as np
from completejourney_py import get_data

print("Fetching transaction data...")
transactions = get_data()['transactions']

# 1. Group the data to the Basket level
# We want the total sales and total discounts for every unique shopping trip
baskets = transactions.groupby('basket_id').agg(
    Total_Sales_Value=('sales_value', 'sum'),
    Retail_Discount=('retail_disc', 'sum'),
    Coupon_Discount=('coupon_disc', 'sum'),
    Coupon_Match=('coupon_match_disc', 'sum'),
    Total_Items=('quantity', 'sum')
).reset_index()

# 2. Calculate the Absolute Total Discount
# In this dataset, discounts are recorded as negative numbers, so we take the absolute value
baskets['Total_Discount'] = (
    baskets['Retail_Discount'].abs() +
    baskets['Coupon_Discount'].abs() +
    baskets['Coupon_Match'].abs()
)

# 3. Calculate the Gross Value of the basket (what it would have cost without any sales/coupons)
baskets['Gross_Value'] = baskets['Total_Sales_Value'] + baskets['Total_Discount']

# 4. Calculate the Discount Ratio
# We use np.where to avoid division by zero for $0 baskets
baskets['Discount_Ratio'] = np.where(
    baskets['Gross_Value'] > 0,
    baskets['Total_Discount'] / baskets['Gross_Value'],
    0
)

# 5. Let's observe the distribution to make a data-driven decision
print("\n📊 Statistical Distribution of the Discount Ratio across all baskets:")
percentiles = [0.25, 0.50, 0.75, 0.85, 0.90, 0.95, 0.99]
display(baskets['Discount_Ratio'].describe(percentiles=percentiles))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.6/31.6 MB 33.8 MB/s eta 0:00:00
Fetching transaction data...

📊 Statistical Distribution of the Discount Ratio across all baskets:


,Discount_Ratio
count,155848.000000
mean,0.133888
std,0.120011
min,0.000000
25%,0.036643
50%,0.113183
75%,0.202684
85%,0.253880
90%,0.294833
95%,0.365931


## Explanation:
Looking directly at the 50th percentile (the median). The average shopper saves about 11% on their grocery bill. Even at the 75th percentile, customers are only saving 20%. This represents normal, healthy promotional behavior.

The anomaly is at the 95th percentile.

These shoppers are securing a 36% to 49% discount across their entire transaction. Because grocery margins rarely exceed 3%, any basket with a total discount ratio above 35% is mathematically guaranteed to be a loss for the store.

We now have our data-driven boundary. We will label any basket with a Discount_Ratio > 0.35 as a Parasitic (1) transaction. Everything else is Profitable (0).

## Step 2: Feature Engineering & Target Definition

Based on the statistical distribution of historical transactions, the median discount ratio is 11%, representing normal shopping behavior. However, the top 5% of baskets exhibit discount ratios exceeding 36%.

Given standard retail margins (2-3%), we establish a strict boundary: **Any basket with a discount ratio > 35% is labeled as Parasitic (1).**

To train our models to identify these baskets without explicitly giving them the discount ratio, we engineer structural basket features:
* **Basket_Size:** Total number of items purchased.
* **Gross_Value:** The total pre-discount value of the basket.
* **Avg_Item_Value:** `Gross_Value / Basket_Size` (Cherry-pickers often target high-value, specific items rather than bulk cheap goods).

In [2]:
# 1. Define the Target Variable (Y)
parasite_threshold = 0.35
baskets['Is_Parasite'] = (baskets['Discount_Ratio'] > parasite_threshold).astype(int)

# 2. Engineer the Features (X)
# We want features that describe the structure of the basket, independent of the discounts
baskets['Avg_Item_Value'] = baskets['Gross_Value'] / baskets['Total_Items']

# Filter out erroneous data (e.g., $0 baskets or negative items)
analytical_df = baskets[(baskets['Gross_Value'] > 0) & (baskets['Total_Items'] > 0)].copy()

# 3. Select the final columns for the machine learning pipeline
features = ['Total_Items', 'Gross_Value', 'Avg_Item_Value']
target = 'Is_Parasite'

ml_df = analytical_df[features + [target]].dropna()

print("✅ Dataset Engineered!")
print(f"Total Baskets Processed: {len(ml_df):,}\n")

# 4. Check the Class Balance
class_balance = ml_df[target].value_counts(normalize=True) * 100
print("⚖️ Class Distribution:")
print(f"Profitable (0): {class_balance[0]:.2f}%")
print(f"Parasitic (1):  {class_balance[1]:.2f}%")

✅ Dataset Engineered!
Total Baskets Processed: 155,381

⚖️ Class Distribution:
Profitable (0): 94.22%
Parasitic (1):  5.78%


## Step 3: Cost-Sensitive Learning & Ensemble Arbitration

The raw dataset exhibits a 94/6 class imbalance. Instead of artificially undersampling and losing valuable data, we embrace the reality of the dataset using **Cost-Sensitive Learning**. We inject class weights into our algorithms, mathematically penalizing them more heavily for missing a "Parasite" than for misclassifying a "Profitable" basket.

Because Support Vector Machines and K-Nearest Neighbors are distance-based algorithms, we apply a `StandardScaler` to ensure features like `Total_Items` and `Gross_Value` share a normalized scale.

Finally, we deploy a **Hard Voting Classifier**. The transaction is evaluated simultaneously by:
1. **SVM (RBF Kernel):** Maps non-linear boundaries in high-dimensional space.
2. **KNN (Distance):** Identifies behavioral proximity to known historical cherry-pickers.
3. **Decision Tree:** Applies strict, hierarchical boolean logic.

A transaction is only classified as a margin-bleeding "Parasite" if the majority of these distinct architectures agree, protecting legitimate shoppers from false positives.

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import VotingClassifier
from sklearn.metrics import classification_report

# 1. Preserve the 94/6 Reality, but take a computational sample so the SVM doesn't crash your RAM
# We use stratify to ensure the 94/6 ratio remains perfectly intact in the sample
sampled_df = ml_df.groupby('Is_Parasite', group_keys=False).apply(lambda x: x.sample(frac=0.1, random_state=42))

# 2. Split Features (X) and Target (y)
X = sampled_df[['Total_Items', 'Gross_Value', 'Avg_Item_Value']]
y = sampled_df['Is_Parasite']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 3. Feature Scaling (Mandatory for SVM and KNN)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 4. Initialize the Individual Defense Layers (Injecting Cost-Sensitive Penalties)
# Note: KNN does not accept class_weights natively, so it relies on the density of the real data
svm_model = SVC(kernel='rbf', class_weight='balanced', probability=True, random_state=42)
knn_model = KNeighborsClassifier(n_neighbors=5)
dt_model = DecisionTreeClassifier(max_depth=5, class_weight='balanced', random_state=42)

# 5. Initialize the Hard Voting Ensemble
ensemble_model = VotingClassifier(
    estimators=[
        ('Support_Vector', svm_model),
        ('K_Nearest', knn_model),
        ('Decision_Tree', dt_model)
    ],
    voting='hard'
)

# 6. Train the Ensemble on the highly imbalanced reality
print("⚙️ Training the Cost-Sensitive Voting Ensemble...")
ensemble_model.fit(X_train_scaled, y_train)
print("✅ Training Complete!\n")

# 7. Evaluate the Arbitration
y_pred = ensemble_model.predict(X_test_scaled)
print("📊 Ensemble Performance Report (Unmanipulated Data):")
print(classification_report(y_test, y_pred, target_names=['Profitable (0)', 'Parasitic (1)']))

/tmp/ipykernel_2123/1563817479.py:11: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  sampled_df = ml_df.groupby('Is_Parasite', group_keys=False).apply(lambda x: x.sample(frac=0.1, random_state=42))


⚙️ Training the Cost-Sensitive Voting Ensemble...
✅ Training Complete!

📊 Ensemble Performance Report (Unmanipulated Data):
                precision    recall  f1-score   support

Profitable (0)       0.96      0.69      0.80      2928
 Parasitic (1)       0.10      0.56      0.17       180

      accuracy                           0.68      3108
     macro avg       0.53      0.62      0.48      3108
  weighted avg       0.91      0.68      0.76      3108

